# CSV to JSON Converter (Fixed)

**Purpose:** Properly convert semicolon-delimited CSV to clean JSON

**Fix:** Use semicolon as delimiter instead of comma


In [ ]:
# ===== CONFIGURATION =====
INPUT_CSV = "/Volumes/dev_automotive/landing/landing_raw/catalogs.csv"
OUTPUT_JSON = "/Volumes/dev_automotive/landing/landing_raw/catalogs_fixed.json"

# CSV delimiter (change this based on your file)
DELIMITER = ";"  # ← THIS IS THE KEY FIX


In [ ]:
import csv
import json
from datetime import datetime

def csv_to_json_fixed(csv_file, json_file, delimiter=";"):
    """
    Convert CSV to JSON with proper delimiter handling

    Args:
        csv_file: Path to input CSV
        json_file: Path to output JSON
        delimiter: CSV delimiter (default: semicolon)

    Returns:
        Number of records converted
    """
    
    print(f"🔄 Converting CSV to JSON...")
    print(f"   Input:  {csv_file}")
    print(f"   Output: {json_file}")
    print(f"   Delimiter: '{delimiter}'")
    print()
    
    try:
        # Read CSV with correct delimiter
        with open(csv_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter=delimiter)
            data = list(reader)
        
        if not data:
            print("❌ No data in file")
            return 0
        
        print(f"✅ Read {len(data):,} records")
        print(f"✅ Columns: {len(data[0])} fields")
        print()
        
        # Show sample of columns
        print("📋 Column names:")
        for i, col in enumerate(list(data[0].keys())[:10], 1):
            print(f"   {i:2d}. {col}")
        if len(data[0]) > 10:
            print(f"   ... and {len(data[0]) - 10} more")
        print()
        
        # Add audit columns
        loaded_at = datetime.utcnow().isoformat() + "Z"
        for row in data:
            row["loaded_at"] = loaded_at
            row["source_file"] = csv_file
        
        # Write JSON with proper formatting
        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"✅ Successfully wrote {len(data):,} records to JSON")
        print(f"✅ File: {json_file}")
        
        return len(data)
    
    except FileNotFoundError:
        print(f"❌ File not found: {csv_file}")
        raise
    except Exception as e:
        print(f"❌ Error: {e}")
        raise


## Run Conversion


In [ ]:
# Convert
record_count = csv_to_json_fixed(INPUT_CSV, OUTPUT_JSON, delimiter=DELIMITER)

print()
print("="*70)
print("CONVERSION COMPLETE")
print("="*70)
print(f"Records: {record_count:,}")


## Verify Output


In [ ]:
print("🔍 Verifying output...")
print()

# Read back and check
import json

with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ JSON is valid")
print(f"✅ Records: {len(data):,}")
print()

# Show first record structure
if data:
    print("📋 Sample record structure:")
    first_record = data[0]
    for key, value in list(first_record.items())[:10]:
        print(f"   {key:30s}: {str(value)[:50]}")
    if len(first_record) > 10:
        print(f"   ... and {len(first_record) - 10} more fields")

print()
print("✅ Output looks good!")


## Test with Spark


In [ ]:
print("🧪 Testing with Spark...")
print()

# Read with Spark
df = spark.read.json(OUTPUT_JSON)

print(f"✅ Spark read successful")
print(f"✅ Records: {df.count():,}")
print(f"✅ Columns: {len(df.columns)}")
print()

print("Schema:")
df.printSchema()

print()
print("Sample data:")
display(df.limit(5))


## Summary

**Before:** Semicolons concatenated in JSON keys/values  
**After:** Proper JSON with separate fields

**Key Fix:** Used `delimiter=";"` in csv.DictReader

**Next Steps:**
- Use this fixed JSON in Bronze ingestion
- No parsing needed in Bronze/Silver
- Clean structure from the start
